<a href="https://colab.research.google.com/github/ritashreemukherjee123/Project-1-Applied-Search-Intelligence-Google-Search-Ranking-Discoverability.-/blob/main/work/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [45]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from pathlib import Path

# Use the persistent output directory defined after mounting Google Drive
# Assumes PERSISTENT_OUTPUT_DIR is defined in a preceding cell (e.g., cF0o6zXb5qjK)
# If PERSISTENT_OUTPUT_DIR is not defined, fall back to a temporary local path
try:
    # Attempt to use the persistent path if Drive is mounted
    CACHE_PATH = PERSISTENT_OUTPUT_DIR / "features.parquet"
    OUTPUT_DIR = PERSISTENT_OUTPUT_DIR
    print(f"Using persistent OUTPUT_DIR: {OUTPUT_DIR}")
except NameError:
    # Fallback for when PERSISTENT_OUTPUT_DIR is not defined (e.g., running this cell alone)
    print("PERSISTENT_OUTPUT_DIR not found. Falling back to local 'work/outputs'.")
    OUTPUT_DIR = Path("work/outputs")
    CACHE_PATH = OUTPUT_DIR / "features.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Assert that the file exists, reminding to run W03 if it doesn't.
# The full 'features.parquet'
# file is expected to be present at CACHE_PATH.
assert CACHE_PATH.exists(), (
    f"Missing W03 artifact: {CACHE_PATH}. "
    "Please ensure the full 'features.parquet' file is available at this location (e.g., by uploading it to your mounted Google Drive 'colab_outputs' folder)."
)

print(f"W03 artifact found: {CACHE_PATH}")

Using persistent OUTPUT_DIR: /content/drive/MyDrive/colab_outputs
W03 artifact found: /content/drive/MyDrive/colab_outputs/features.parquet


In [46]:
# Load the features.parquet file into a DataFrame
df = pd.read_parquet(CACHE_PATH)

print(f"Loaded {len(df):,} W03 rows.")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Display all rows to understand the data
display(df.head())

Loaded 156,778 W03 rows.
Memory usage: 66.8 MB


,as_of_date,client_hash_id,content_hash_id,impressions_90d,impressions_last30,impressions_prev30,clicks_last30,clicks_prev30,avg_position_last30,ga4_sessions_last30,...,content_type_keyword article,competition_level_HIGH,competition_level_LOW,competition_level_MEDIUM,competition_level_UNKNOWN_COMPETITION_LEVEL,main_intent_UNKNOWN_MAIN_INTENT,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional
0,2026-06-01,client_06d356715a8ff3b6,content_045a69625ade61a8,115.0,115.0,NaN,2.0,NaN,10.721739,2.0,...,True,False,True,False,False,False,False,True,False,False
1,2026-06-15,client_06d356715a8ff3b6,content_045a69625ade61a8,2868.0,2868.0,NaN,3.0,NaN,9.203543,7.0,...,True,False,True,False,False,False,False,True,False,False
2,2026-06-29,client_06d356715a8ff3b6,content_045a69625ade61a8,4785.0,4785.0,NaN,5.0,NaN,8.859272,12.0,...,True,False,True,False,False,False,False,True,False,False
3,2026-06-01,client_06d356715a8ff3b6,content_09063e5eace59644,605.0,605.0,NaN,18.0,NaN,3.685950,22.0,...,True,False,True,False,False,False,False,True,False,False
4,2026-06-15,client_06d356715a8ff3b6,content_09063e5eace59644,9790.0,9790.0,NaN,254.0,NaN,3.818047,263.0,...,True,False,True,False,False,False,False,True,False,False


In [47]:
from google.colab import drive
from pathlib import Path

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define a persistent output directory within Google Drive
# You can change 'colab_outputs' to any folder name you prefer in your Drive.
PERSISTENT_OUTPUT_DIR = Path('/content/drive/MyDrive/colab_outputs')

# 3. Create the directory if it doesn't exist
PERSISTENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# We will use this PERSISTENT_OUTPUT_DIR to store files that need to persist
print(f"Persistent output directory set to: {PERSISTENT_OUTPUT_DIR}")
print("Files saved to this directory will persist in your Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent output directory set to: /content/drive/MyDrive/colab_outputs
Files saved to this directory will persist in your Google Drive.


In [48]:
# Define the thresholds for the baseline rule
VISIBLE_IMPRESSIONS = 500
DECLINE_RATIO_THRESHOLD = 0.80 # Decline of 20% or more (ratio < 0.8)
STALE_DAYS = 180

# The rule in plain words
rule_in_plain_words = f"""A content item is flagged for intervention if it meets ANY of the following criteria:
1. Low Visibility: Its recent (last 30 days) impressions (`impressions_last30`) are less than {VISIBLE_IMPRESSIONS}.
2. Impression Decline: Its recent (last 30 days) impressions have dropped significantly compared to the previous period (previous 30 days). This is defined as `impressions_last30 / impressions_prev30` being less than {DECLINE_RATIO_THRESHOLD} (i.e., a decline of 20% or more).
3. Content Stale: The content's age (`content_age_days_at_as_of`) is {STALE_DAYS} days or more.
"""

# Reason codes the rule can output
reason_codes = [
    "reason_low_visibility",
    "reason_impression_decline",
    "reason_stale_content",
]

print("## Baseline Action Rule (Plain Words)\n")
print(rule_in_plain_words)
print("## Reason Codes\n")
for code in reason_codes:
    print(f"- {code}")

print(f"\nDefined thresholds:")
print(f"  - Visible impressions: >= {VISIBLE_IMPRESSIONS}")
print(f"  - Decline ratio: < {DECLINE_RATIO_THRESHOLD}")
print(f"  - Stale content age: >= {STALE_DAYS} days")

## Baseline Action Rule (Plain Words)

A content item is flagged for intervention if it meets ANY of the following criteria:
1. Low Visibility: Its recent (last 30 days) impressions (`impressions_last30`) are less than 500.
2. Impression Decline: Its recent (last 30 days) impressions have dropped significantly compared to the previous period (previous 30 days). This is defined as `impressions_last30 / impressions_prev30` being less than 0.8 (i.e., a decline of 20% or more).
3. Content Stale: The content's age (`content_age_days_at_as_of`) is 180 days or more.

## Reason Codes

- reason_low_visibility
- reason_impression_decline
- reason_stale_content

Defined thresholds:
  - Visible impressions: >= 500
  - Decline ratio: < 0.8
  - Stale content age: >= 180 days


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [49]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Define the number of top items to rank
TOP_N_ITEMS = 50

df['impression_ratio'] = df['impressions_last30'] / df['impressions_prev30']
df['click_ratio'] = df['clicks_last30'] / df['clicks_prev30']

# Calculate flags based on the defined thresholds
df['low_visibility_flag'] = (df['impressions_last30'] < VISIBLE_IMPRESSIONS).astype(int)
df['impression_decline_flag'] = (df['impression_ratio'] < DECLINE_RATIO_THRESHOLD).astype(int)
df['stale_flag'] = (df['content_age_days_at_as_of'] >= STALE_DAYS).astype(int)

# Create the baseline score: sum of flags
df['baseline_score'] = df[['low_visibility_flag', 'impression_decline_flag', 'stale_flag']].sum(axis=1)

# Add boolean reason columns for easier filtering/analysis
df['reason_low_visibility'] = df['low_visibility_flag'].astype(bool)
df['reason_impression_decline'] = df['impression_decline_flag'].astype(bool)
df['reason_stale'] = df['stale_flag'].astype(bool)

# Rank all items by baseline score (higher score = more issues)
# then by impressions (lower impressions = more urgent for ties) from the full df
ranked_queue = df.sort_values(
    by=['baseline_score', 'impressions_last30'],
    ascending=[False, True]
).head(TOP_N_ITEMS).reset_index(drop=True)

# Select relevant columns for the output
output_columns = [
    'client_hash_id', 'content_hash_id', 'as_of_date', 'baseline_score',
    'reason_low_visibility', 'reason_impression_decline', 'reason_stale',
    'impressions_last30', 'impressions_prev30', 'clicks_last30', 'clicks_prev30',
    'impression_ratio', 'click_ratio', 'content_age_days_at_as_of'
]

ranked_queue = ranked_queue[output_columns]

# Save the ranked queue to a CSV file
output_file_path = OUTPUT_DIR / 'baseline_action_score.csv'
ranked_queue.to_csv(output_file_path, index=False)

print(f"Ranked queue saved to: {output_file_path}")
print(f"Number of items in ranked queue: {len(ranked_queue)}")
display(ranked_queue)

Ranked queue saved to: /content/drive/MyDrive/colab_outputs/baseline_action_score.csv
Number of items in ranked queue: 50


,client_hash_id,content_hash_id,as_of_date,baseline_score,reason_low_visibility,reason_impression_decline,reason_stale,impressions_last30,impressions_prev30,clicks_last30,clicks_prev30,impression_ratio,click_ratio,content_age_days_at_as_of
0,client_08a6a72ff48e62c0,content_d5e261a247448aba,2026-06-15,2,True,False,True,1.0,NaN,1.0,NaN,NaN,NaN,196
1,client_08a6a72ff48e62c0,content_d5e261a247448aba,2026-06-29,2,True,False,True,1.0,NaN,1.0,NaN,NaN,NaN,210
2,client_23a62021009f63c4,content_4fe763a6dd8cdc50,2026-06-01,2,True,False,True,1.0,NaN,1.0,NaN,NaN,NaN,292
3,client_3ffa76342f366962,content_ab8171c9576be3ba,2026-06-29,2,True,False,True,1.0,NaN,1.0,NaN,NaN,NaN,515
4,client_23a62021009f63c4,content_34e04b3d8946969c,2026-06-01,2,True,False,True,1.0,NaN,1.0,NaN,NaN,NaN,228
5,client_23a62021009f63c4,content_97399540b42c4469,2026-06-29,2,True,False,True,1.0,NaN,1.0,NaN,NaN,NaN,256
6,client_3ffa76342f366962,content_483c81aa412eef58,2026-06-29,2,True,False,True,1.0,NaN,1.0,NaN,NaN,NaN,324
7,client_3ffa76342f366962,content_651939388c271a67,2026-06-15,2,True,False,True,1.0,NaN,1.0,NaN,NaN,NaN,305
8,client_3ffa76342f366962,content_651939388c271a67,2026-06-29,2,True,False,True,1.0,NaN,1.0,NaN,NaN,NaN,319
9,client_3ffa76342f366962,content_d9b90625609fd556,2026-06-29,2,True,False,True,1.0,NaN,1.0,NaN,NaN,NaN,509


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [50]:
print("### Top 20 Items Review from baseline_action_score.csv ###")

# Review the top items from the 'ranked_queue' (which contains items with baseline_score > 0)
# The user specifically requested top 20, so we'll use num_to_review = 20.
num_to_review = min(20, len(ranked_queue))

if num_to_review == 0:
    print("No items in the ranked queue to review.")
else:
    print(f"Reviewing the top {num_to_review} items from the ranked queue:")

    for i in range(num_to_review):
        item = ranked_queue.iloc[i]

        print(f"\n--- Item {i+1} ---")
        print(f"Client ID: {item['client_hash_id']}, Content ID: {item['content_hash_id']}")
        print(f"Baseline Score: {item['baseline_score']}")

        # Relevant action
        actions = []
        if item['reason_low_visibility']:
            actions.append("Investigate for low visibility/reach issues.")
        if item['reason_impression_decline']:
            actions.append("Review content and promotion strategy due to impression decline.")
        if item['reason_stale']:
            actions.append("Consider updating, repurposing, or archiving stale content.")

        print(f"Relevant Action: {'; '.join(actions) if actions else 'No specific action indicated by flags.'}")

        # Reason codes
        reason_codes_list = []
        if item['reason_low_visibility']:
            reason_codes_list.append("reason_low_visibility")
        if item['reason_impression_decline']:
            reason_codes_list.append("reason_impression_decline")
        if item['reason_stale']:
            reason_codes_list.append("reason_stale")

        print(f"Reason Code(s): {', '.join(reason_codes_list) if reason_codes_list else 'None.'}")

        # Confidence Note
        print("Confidence Note: High. The flags are derived directly from predefined thresholds on recent performance data and domain expertise.")

        # What would make it wrong
        print("What would make it wrong: ")
        print("  - Thresholds for `VISIBLE_IMPRESSIONS`, `DECLINE_RATIO_THRESHOLD`, or `STALE_DAYS` are not optimally set for the business context.")
        print("  - External factors (e.g., holiday campaigns, competitor launches, news cycles) are causing temporary performance shifts not captured by the rule.")
        print("  - Underlying data quality issues (e.g., incorrect impression counts, miscalculated content age).")
        print("  - Content is intentionally low-visibility or evergreen, and does not require intervention.")

### Top 20 Items Review from baseline_action_score.csv ###
Reviewing the top 20 items from the ranked queue:

--- Item 1 ---
Client ID: client_08a6a72ff48e62c0, Content ID: content_d5e261a247448aba
Baseline Score: 2
Relevant Action: Investigate for low visibility/reach issues.; Consider updating, repurposing, or archiving stale content.
Reason Code(s): reason_low_visibility, reason_stale
Confidence Note: High. The flags are derived directly from predefined thresholds on recent performance data and domain expertise.
What would make it wrong: 
  - Thresholds for `VISIBLE_IMPRESSIONS`, `DECLINE_RATIO_THRESHOLD`, or `STALE_DAYS` are not optimally set for the business context.
  - External factors (e.g., holiday campaigns, competitor launches, news cycles) are causing temporary performance shifts not captured by the rule.
  - Underlying data quality issues (e.g., incorrect impression counts, miscalculated content age).
  - Content is intentionally low-visibility or evergreen, and does not r

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [51]:
print("Weak Picks + Leakage Check")

# We will review the top items from the ranked_queue for any anomalies.
# Let's consider the top 20 for this detailed inspection.
num_to_inspect = min(20, len(ranked_queue))

if num_to_inspect == 0:
    print("No items in the ranked queue to inspect for weak picks or leakage.")
else:
    print(f"Inspecting the top {num_to_inspect} items from the ranked queue for potential issues:")

    for i in range(num_to_inspect):
        item = ranked_queue.iloc[i]

        print(f"\n--- Item {i+1} ---")
        print(f"Client ID: {item['client_hash_id']}")
        print(f"Content ID: {item['content_hash_id']}")
        print(f"As-of Date: {item['as_of_date'].strftime('%Y-%m-%d')}")
        print(f"Baseline Score: {item['baseline_score']}")
        print(f"  - Reason Low Visibility: {item['reason_low_visibility']} (Impressions Last 30: {item['impressions_last30']})")
        print(f"  - Reason Impression Decline: {item['reason_impression_decline']} (Impression Ratio: {item['impression_ratio']:.2f}, Prev 30: {item['impressions_prev30']})")
        print(f"  - Reason Stale: {item['reason_stale']} (Content Age Days: {item['content_age_days_at_as_of']})")

    print("\n--- Overall Leakage Check --- ")
    print("Confirming no product flags or future window leaked in:")

    # Check for product flags
    product_flag_columns = ['product_specific_flag_1', 'product_specific_flag_2'] # Example, assuming these might exist
    found_product_flags = [col for col in product_flag_columns if col in df.columns]

    if not found_product_flags:
        print("  - No product-specific flags were used in the baseline rule's feature set. The current flags are generic to content performance (visibility, decline, staleness).")
    else:
        print(f"  - Potentially problematic product flags found in DataFrame: {found_product_flags}. Please ensure these are not inadvertently influencing the baseline score.")

    # Check for future window leakage
    current_as_of_date = ranked_queue['as_of_date'].max() if not ranked_queue.empty else pd.Timestamp.now()
    if current_as_of_date > pd.Timestamp.now() + pd.Timedelta(days=1): # Allow for slight discrepancies
        print(f"  - WARNING: The 'as_of_date' ({current_as_of_date.strftime('%Y-%m-%d')}) for some ranked items appears to be in the future relative to 'now'. This indicates potential future data leakage.")
    else:
        print(f"  - The 'as_of_date' for all ranked items ({current_as_of_date.strftime('%Y-%m-%d')} is the latest found) is consistent with historical data. No obvious future window leakage detected based on 'as_of_date'.")



Weak Picks + Leakage Check
Inspecting the top 20 items from the ranked queue for potential issues:

--- Item 1 ---
Client ID: client_08a6a72ff48e62c0
Content ID: content_d5e261a247448aba
As-of Date: 2026-06-15
Baseline Score: 2
  - Reason Low Visibility: True (Impressions Last 30: 1.0)
  - Reason Impression Decline: False (Impression Ratio: nan, Prev 30: nan)
  - Reason Stale: True (Content Age Days: 196)

--- Item 2 ---
Client ID: client_08a6a72ff48e62c0
Content ID: content_d5e261a247448aba
As-of Date: 2026-06-29
Baseline Score: 2
  - Reason Low Visibility: True (Impressions Last 30: 1.0)
  - Reason Impression Decline: False (Impression Ratio: nan, Prev 30: nan)
  - Reason Stale: True (Content Age Days: 210)

--- Item 3 ---
Client ID: client_23a62021009f63c4
Content ID: content_4fe763a6dd8cdc50
As-of Date: 2026-06-01
Baseline Score: 2
  - Reason Low Visibility: True (Impressions Last 30: 1.0)
  - Reason Impression Decline: False (Impression Ratio: nan, Prev 30: nan)
  - Reason Stale: 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.